# Week 3 lab: find what the file is hiding

A file lands on your desk. Someone will make a decision with it at the next
shift handover. Your job today is not to clean it. Your job is to find out what
it will not support, and to leave evidence another person can check.

Three rules hold for the whole lab:

1. **Never edit the raw file.** A fault you delete is a fault nobody else can see.
2. **Report, do not repair.** A correction needs a source, an approver, and a record.
3. **Name your evidence.** "Row 44" beats "some temperatures look high".

Your file is generated from your username, so the faults in it are yours. The
person next to you has different ones. Comparing answers is useful; copying them
will not work.

## Setup


```bash
python src/make_variant.py --student-id your-github-username
```

Then set `STUDENT_ID` below to the sameRun this in the terminal first, using your own GitHub Classroom username:
 value.

In [4]:
from pathlib import Path
import hashlib
import sys

import pandas as pd
from IPython.display import display

# Put your GitHub Classroom username here. Your file is generated from it.
STUDENT_ID = "i-aroojfatimah"


def locate_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "validate_data.py").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from the week03 project folder.")


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()


PROJECT_ROOT = locate_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from make_variant import slugify  # noqa: E402

DATA_PATH = PROJECT_ROOT / "data" / "raw" / f"shift_log_{slugify(STUDENT_ID)}.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH.name} is missing. Run this in the terminal first:\n"
        f"    python src/make_variant.py --student-id {STUDENT_ID}"
    )

df = pd.read_csv(DATA_PATH)
print(f"File:    {DATA_PATH.name}")
print(f"Rows:    {len(df)}   Columns: {len(df.columns)}")
print(f"SHA-256: {file_sha256(DATA_PATH)}")
print("\nCopy the file name and fingerprint into docs/data_quality_report.md now.")

File:    shift_log_i-aroojfatimah.csv
Rows:    73   Columns: 8
SHA-256: 05525440ad4db1ff572ebd2218851e57569fcde4957c43e353e223881d8f036a

Copy the file name and fingerprint into docs/data_quality_report.md now.


## Step 1. Write the contract before you look

Before any code, answer these in the cell below. You are writing the standard
that the rest of the lab measures the file against. If you skip this, every
check you write later is just an opinion.

- What does one row represent?
- Which columns together identify one row uniquely?
- How many records should each line have, and over what period?
- Which columns can never be negative, and why is that true from their meaning
  rather than from the numbers you happen to see?

**Your contract:**

- One row represents:
- The business key is:
- Expected coverage:
- Columns that cannot be negative, and why:

one production line in one hourly period.
line_id + timestamp
.each of the three lines (Line_A, Line_B, Line_C) should have 24 hourly records over one simulated day.
units_produced, downtime_min, energy_kwh, and rejected_units cannot be negative because they represent counts, duration, energy consumption, and rejected-item counts.

In [5]:
display(df.head(6))

schema = pd.DataFrame({
    "column": df.columns,
    "dtype_on_load": [str(value) for value in df.dtypes],
    "missing": [int(df[column].isna().sum()) for column in df.columns],
    "distinct": [int(df[column].nunique(dropna=True)) for column in df.columns],
})
display(schema)

,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
0,2026-01-12T06:00,Line_A,Day,389,10.7,160.5,5,77.1
1,2026-01-12T06:00,Line_B,Day,359,NaN,146.6,12,75.3
2,2026-01-12T06:00,Line_C,Day,387,3.8,170.6,3,74.2
3,2026-01-12T07:00,Line_A,Day,397,14.9,178.4,7,74.3
4,2026-01-12T07:00,Line_B,Day,391,12.5,161.9,7,75.4
5,2026-01-12T07:00,Line_C,Day,397,11.6,171.5,3,74.3


,column,dtype_on_load,missing,distinct
0,timestamp,str,0,25
1,line_id,str,0,3
2,shift,str,0,2
3,units_produced,int64,0,47
4,downtime_min,float64,1,61
5,energy_kwh,float64,0,66
6,rejected_units,int64,0,14
7,motor_temp_c,float64,0,37


## Step 2. Hunt, before the validator helps you

Work through the four cells below **without running the validator**. You are
looking for records you would not want a supervisor to act on.

Keep a list as you go. You will be asked to commit to it in a moment.

### Completeness

In [6]:
# Completeness: find missing values and show the affected records.

missing_counts = df.isna().sum()
print("Missing values per column:")
display(missing_counts[missing_counts > 0])

missing_rows = df[df.isna().any(axis=1)]

print("\nRows containing missing values:")
display(missing_rows)

Missing values per column:


downtime_min    1
dtype: int64


Rows containing missing values:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
1,2026-01-12T06:00,Line_B,Day,359,NaN,146.6,12,75.3


### Uniqueness

In [7]:
# Uniqueness: check exact duplicate rows and duplicate line-hours.

# 1. Exact duplicate records
exact_duplicates = df[df.duplicated(keep=False)]

print("Exact duplicate records:")
display(exact_duplicates)

# 2. Same line + same timestamp appearing more than once
line_hour_counts = (
    df.groupby(["line_id", "timestamp"], dropna=False)
      .size()
      .reset_index(name="count")
)

duplicate_line_hours = line_hour_counts[line_hour_counts["count"] > 1]

print("\nLine-hours appearing more than once:")
display(duplicate_line_hours)

# Show the actual records for duplicate line-hours
if not duplicate_line_hours.empty:
    duplicate_keys = duplicate_line_hours[["line_id", "timestamp"]]
    duplicate_records = df.merge(
        duplicate_keys,
        on=["line_id", "timestamp"],
        how="inner"
    )
    print("\nActual records for duplicate line-hours:")
    display(duplicate_records)

Exact duplicate records:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
27,2026-01-12T15:00,Line_A,Day,442,4.2,158.5,4,76.4
72,2026-01-12T15:00,Line_A,Day,442,4.2,158.5,4,76.4



Line-hours appearing more than once:


,line_id,timestamp,count
10,Line_A,2026-01-12T15:00,2



Actual records for duplicate line-hours:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
0,2026-01-12T15:00,Line_A,Day,442,4.2,158.5,4,76.4
1,2026-01-12T15:00,Line_A,Day,442,4.2,158.5,4,76.4


### Validity

Separate two ideas that feel similar and are not:

- **impossible** under the contract, such as rejecting more units than you made;
- **surprising**, such as a high motor temperature, which may be an error, a
  real event, or a line running differently.

Only the first kind can be judged from the file alone.

In [8]:
# Validity: inspect ranges and separate impossible values from unusual ones.

print("Descriptive statistics:")
display(df.describe(include="all"))

print("\nImpossible under the data contract:")

# Values that cannot be negative by meaning.
negative_values = df[
    (df["units_produced"] < 0)
    | (df["downtime_min"] < 0)
    | (df["energy_kwh"] < 0)
    | (df["rejected_units"] < 0)
]

display(negative_values)

print("\nSurprising but possible:")

# High motor temperature is unusual, but without an approved
# engineering limit it cannot automatically be called impossible.
high_temperature = df[df["motor_temp_c"] > 100]

display(high_temperature)

Descriptive statistics:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
count,73,73,73,73.000000,72.000000,73.000000,73.000000,73.000000
unique,25,3,2,NaN,NaN,NaN,NaN,NaN
top,2026-01-12T15:00,Line_A,Day,NaN,NaN,NaN,NaN,NaN
freq,4,25,38,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,399.000000,9.966667,159.880822,6.493151,75.600000
std,NaN,NaN,NaN,28.537013,5.369213,22.965836,3.543865,5.237154
min,NaN,NaN,NaN,348.000000,1.000000,-14.900000,2.000000,72.000000
25%,NaN,NaN,NaN,379.000000,6.350000,156.100000,4.000000,74.300000
50%,NaN,NaN,NaN,397.000000,9.300000,162.700000,5.000000,75.200000
75%,NaN,NaN,NaN,425.000000,13.450000,168.700000,8.000000,75.800000



Impossible under the data contract:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
20,2026-01-12T12:00,Line_C,Day,423,7.3,-14.9,6,75.5



Surprising but possible:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
6,2026-01-12T08:00,Line_A,Day,432,10.0,166.8,5,118.3


### Coverage and time

In [9]:
# Coverage and time checks

parsed_timestamp = pd.to_datetime(df["timestamp"], errors="coerce")

print("Invalid timestamps:")
display(df[parsed_timestamp.isna()])

print("\nUnique line-hours:")
line_hours = (
    df.assign(parsed_timestamp=parsed_timestamp)
      .groupby("line_id")["parsed_timestamp"]
      .nunique()
)

display(line_hours)

print("\nLines with fewer or more than 24 distinct hours:")
display(line_hours[line_hours != 24])

print("\nShift labels that disagree with the hour:")

hour = parsed_timestamp.dt.hour

expected_shift = hour.map(
    lambda h: "Day" if 6 <= h <= 17 else "Night"
)

shift_mismatch = df[
    parsed_timestamp.notna()
    & df["shift"].ne(expected_shift)
]

display(shift_mismatch)

Invalid timestamps:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
66,13/01/2026 04:00,Line_A,Night,392,7.8,173.3,7,73.0



Unique line-hours:


line_id
Line_A    23
Line_B    24
Line_C    24
Name: parsed_timestamp, dtype: int64


Lines with fewer or more than 24 distinct hours:


line_id
Line_A    23
Name: parsed_timestamp, dtype: int64


Shift labels that disagree with the hour:


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c
43,2026-01-12T20:00,Line_B,Day,401,15.0,157.4,12,75.8


## Step 3. Commit to your list

**Stop here and fill in section 3 of `docs/data_quality_report.md`** with the
records you found, before running the next cell.

This is the point of the exercise. Writing your list down first is what makes
the next cell informative rather than reassuring.

## Step 4. Compare against the checks

The validator runs the rules that are written so far. Unwritten ones are
reported as TODO rather than stopping the run.

In [10]:
from validate_data import validate
payload = validate(DATA_PATH, skip_unwritten=True)
summary = pd.DataFrame(payload["checks"])[["name", "passed", "observed", "affected_rows"]]
display(summary)
print("Not written yet:", payload["unwritten_checks"] or "none")
print("Overall status:", payload["status"])

,name,passed,observed,affected_rows
0,required_columns,True,[],[]
1,timestamp_parse,False,1,[68]
2,required_values_complete,False,1,[3]
3,exact_row_uniqueness,False,2,"[29, 74]"
4,downtime_range,True,0,[]
5,motor_temperature_review_range,False,1,[8]


Not written yet: ['check_business_key_uniqueness', 'check_known_line_ids', 'check_positive_energy', 'check_rejects_within_output', 'check_shift_matches_hour', 'check_expected_line_hour_coverage']
Overall status: FAIL


**Answer before moving on:**

- Which records did the code find that you missed?
- Which did you find that no check covers yet? That is a check you should write.
- Did any single fault trip more than one check? Is that one problem or two?

**Your answer:**

The code did not find any records that I missed. My manual checks identified the same timestamp, missing-value, duplicate, and high-temperature records.
My manual checks identified a negative energy value and a shift-hour mismatch that are not covered by the written checks yet. The line-hour coverage issue also needs a dedicated check.
A single underlying data problem can affect more than one check. For example, a timestamp problem can also affect time-coverage analysis. It should still be treated as one underlying data issue, while documenting each check it affects.

## Step 5. Write the missing checks

Move to the terminal and work in `src/validate_data.py`:

```bash
pytest -q
```

Six checks are marked TODO and six tests are failing. Write one check, re-run,
and repeat until the tests pass. Then come back and run the cell below.

In [ ]:
payload = validate(DATA_PATH)
summary = pd.DataFrame(payload["checks"])[["name", "passed", "observed", "affected_rows"]]
display(summary)
print("Failed checks:", payload["failed_checks"])
print("Overall status:", payload["status"])

assert not payload["unwritten_checks"], "Some checks are still TODO. Run pytest -q."
print("\nAll twelve checks are written. Now run the script from the terminal so it")
print("writes artifacts/validation_results.json, and commit that file.")

## Step 6. The row that checks cannot settle

One record in your file is unusual rather than impossible: the high motor
temperature. A check can flag it. A check cannot tell you what it means.

**Answer below:**

- Name three explanations for that reading, at least one of which is not a fault.
- What single piece of evidence would separate them?
- Who holds that evidence?
- Until you have it, what may the handover discussion still use this file for?

**Your answer:**

## Next

1. Finish `docs/data_quality_report.md` with what you now know.
2. Do the attack round with your partner and fill in `docs/attack_log.md`.
3. Make the approval decision in `docs/dataset_approval.md`.
4. Commit and push.